# 📘 Track B — Notebook 2 (v2): Gaussian Interpolation on Frame-Level Embeddings

**Project:** Synthetic Data Generation for Speech Emotion Recognition  
**Version:** v2 — operates on `(128, 768)` frame-level embeddings

---

## Key Change from v1
Gaussian interpolation now operates **per time frame** rather than on a single flat vector.  
For each group of 5 real embeddings `(5, 128, 768)`, we fit a Gaussian independently at each
of the 128 time positions, then sample new trajectories. This produces synthetic embeddings
that vary realistically across time — critical for good audio reconstruction.

---

## Output
- `synthetic_embeddings/synth_actor*_sent*_*.npy` — shape `(35, 128, 768)` per group
- `synthetic_embeddings/all_synthetic_matrix.npy` — shape `(22400, 128, 768)`
- `synthetic_embeddings/all_synthetic_labels.csv`

---
## CELL 1 — Imports & Environment Check

In [ ]:
import sys, time, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.spatial.distance import cosine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

sns.set_theme(style='whitegrid', palette='muted')

print('=' * 55)
print('   ENVIRONMENT CHECK — NOTEBOOK 2 (v2)')
print('=' * 55)
cuda_available = torch.cuda.is_available()
print(f'  CUDA Available : {cuda_available}')
if cuda_available:
    print(f'  GPU            : {torch.cuda.get_device_name(0)}')
print(f'  Python         : {sys.version.split()[0]}')
print('  Note: Gaussian sampling is CPU-only. GPU used in Notebook 3.')
print('=' * 55)

---
## CELL 2 — Configuration

In [ ]:
EMBEDDINGS_DIR  = Path('./embeddings')
SYNTH_EMB_DIR   = Path('./synthetic_embeddings')
OUTPUTS_DIR     = Path('./outputs')
SYNTH_EMB_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

EMOTIONS    = ['anger', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'sarcastic', 'surprise']
N_ACTORS    = 8
N_SENTENCES = 10
N_SESSIONS  = 5

# Embedding shape (must match Notebook 1 v2)
TARGET_FRAMES = 128
HUBERT_DIM    = 768

# Generation config
N_SYNTHETIC_PER_GROUP = 35    # 5 real + 35 synthetic = 40 per group
ALPHA             = 0.7       # Covariance scaling (0.7 = conservative)
REG_COVAR         = 1e-4      # Diagonal regularization
GAUSSIAN_FRACTION = 0.6       # 60% Gaussian, 40% convex interpolation
RANDOM_SEED       = 42
np.random.seed(RANDOM_SEED)

print('Configuration loaded:')
print(f'  Embedding shape       : ({TARGET_FRAMES}, {HUBERT_DIM})')
print(f'  Synthetic per group   : {N_SYNTHETIC_PER_GROUP}')
print(f'  Alpha                 : {ALPHA}')
print(f'  Total synthetic files : {N_ACTORS * N_SENTENCES * len(EMOTIONS) * N_SYNTHETIC_PER_GROUP}')

# Validate prerequisite files
matrix_path = EMBEDDINGS_DIR / 'embedding_matrix.npy'
assert matrix_path.exists(), '❌ embedding_matrix.npy not found. Run Notebook 1 first.'

# Confirm v2 shape
test_load = np.load(matrix_path)
assert test_load.ndim == 3 and test_load.shape[1:] == (TARGET_FRAMES, HUBERT_DIM), \
    f'❌ Wrong embedding shape {test_load.shape}. Expected (N, {TARGET_FRAMES}, {HUBERT_DIM}). Re-run Notebook 1 v2.'
print(f'\n✅ Embeddings confirmed v2 shape: {test_load.shape}')

---
## CELL 3 — Load Real Embeddings

In [ ]:
emb_matrix = np.load(EMBEDDINGS_DIR / 'embedding_matrix.npy')   # (3200, 128, 768)
label_df   = pd.read_csv(EMBEDDINGS_DIR / 'embedding_labels.csv')

print(f'Embedding matrix shape : {emb_matrix.shape}  ← (N_files, frames, dims)')
print(f'Label DataFrame shape  : {label_df.shape}')

# Load per-group files
emb_files = list(EMBEDDINGS_DIR.glob('actor*_sent*_*.npy'))
assert len(emb_files) > 0, '❌ No per-group .npy files found.'

group_embeddings = {}
for fpath in emb_files:
    match = re.match(r'actor(\d+)_sent(\d+)_(.+)\.npy', fpath.name)
    if match:
        actor, sentence, emotion = int(match.group(1)), int(match.group(2)), match.group(3)
        arr = np.load(fpath)   # (5, 128, 768)
        group_embeddings[(actor, sentence, emotion)] = arr

print(f'\nLoaded {len(group_embeddings)} group arrays')
print(f'Expected: {N_ACTORS * N_SENTENCES * len(EMOTIONS)}')

sample_key = list(group_embeddings.keys())[0]
print(f'\nSpot check — group {sample_key}:')
print(f'  Shape : {group_embeddings[sample_key].shape}  ← should be (5, 128, 768)')
print('\n✅ All embeddings loaded.')

---
## CELL 4 — Frame-Level Gaussian Sampler

**KEY CHANGE FROM v1:**  
For each group `(5, 128, 768)`, we fit a Gaussian **independently at each of the 128 time frames**.  
Sampling produces new full trajectories `(128, 768)` that vary naturally over time.

In [ ]:
def generate_synthetic_embeddings_framelevel(
        real_embeddings: np.ndarray,
        n_total: int,
        alpha: float,
        gaussian_fraction: float,
        reg_covar: float) -> np.ndarray:
    """
    Generate synthetic frame-level embeddings from a group of real ones.

    Args:
        real_embeddings   : (N_real, TARGET_FRAMES, HUBERT_DIM) = (5, 128, 768)
        n_total           : Number of synthetic samples to generate
        alpha             : Covariance scaling
        gaussian_fraction : Fraction generated via Gaussian sampling
        reg_covar         : Covariance regularization

    Returns:
        np.ndarray of shape (n_total, TARGET_FRAMES, HUBERT_DIM)

    Method:
    - Gaussian path: at each time frame t, compute mean and variance across
      the 5 real samples, then sample independently per frame.
      Uses per-dimension variance (diagonal covariance) — avoids singular
      matrix issues with only 5 samples.
    - Convex path: randomly interpolate between two real trajectories.
    """
    N_real, T, D = real_embeddings.shape  # (5, 128, 768)
    n_gaussian   = int(n_total * gaussian_fraction)
    n_convex     = n_total - n_gaussian

    # --- Gaussian sampling (frame-by-frame) ---
    # mean: (T, D), std: (T, D)
    mean_traj = real_embeddings.mean(axis=0)            # (T, D)
    std_traj  = real_embeddings.std(axis=0) + reg_covar # (T, D)  add floor

    # Sample n_gaussian trajectories
    # noise shape: (n_gaussian, T, D)
    noise         = np.random.randn(n_gaussian, T, D).astype(np.float32)
    gauss_samples = mean_traj[np.newaxis] + alpha * std_traj[np.newaxis] * noise
    # gauss_samples: (n_gaussian, T, D)

    # --- Convex interpolation (between two real trajectories) ---
    convex_samples = []
    for _ in range(n_convex):
        i, j = np.random.choice(N_real, size=2, replace=(N_real < 2))
        lam   = np.random.uniform(0.1, 0.9)
        interp = lam * real_embeddings[i] + (1 - lam) * real_embeddings[j]  # (T, D)
        convex_samples.append(interp)

    if convex_samples:
        convex_arr = np.stack(convex_samples)  # (n_convex, T, D)
        return np.concatenate([gauss_samples, convex_arr], axis=0)  # (n_total, T, D)
    else:
        return gauss_samples


print('✅ Frame-level Gaussian sampler defined.')
print(f'   Per group: {int(N_SYNTHETIC_PER_GROUP * GAUSSIAN_FRACTION)} Gaussian '
      f'+ {N_SYNTHETIC_PER_GROUP - int(N_SYNTHETIC_PER_GROUP * GAUSSIAN_FRACTION)} convex')

---
## CELL 5 — Single Group Dry Run (Sanity Check)

In [ ]:
test_key = (1, 1, 'anger')
if test_key not in group_embeddings:
    test_key = list(group_embeddings.keys())[0]
    print(f'Defaulting to: {test_key}')

real_embs  = group_embeddings[test_key]  # (5, 128, 768)
synth_embs = generate_synthetic_embeddings_framelevel(
    real_embs, N_SYNTHETIC_PER_GROUP, ALPHA, GAUSSIAN_FRACTION, REG_COVAR
)  # (35, 128, 768)

print(f'Real shape  : {real_embs.shape}')
print(f'Synth shape : {synth_embs.shape}  ← should be ({N_SYNTHETIC_PER_GROUP}, 128, 768)')

# Quality metrics on flattened versions
real_flat   = real_embs.reshape(len(real_embs), -1)    # (5, 98304)
synth_flat  = synth_embs.reshape(len(synth_embs), -1)  # (35, 98304)
real_mean_f = real_flat.mean(axis=0)

cos_sims = [1 - cosine(s, real_mean_f) for s in synth_flat]
ratio    = synth_flat.std() / real_flat.std()

print(f'\n--- Quality Metrics ---')
print(f'  Cosine similarity (synth vs real mean):')
print(f'    Mean : {np.mean(cos_sims):.4f}  (>0.8 is good)')
print(f'    Min  : {np.min(cos_sims):.4f}')
print(f'  Std ratio (synth/real) : {ratio:.4f}  (close to 1.0 is good)')

# Temporal structure check
real_temporal_std  = real_embs.std(axis=2).mean()   # avg std over dims per frame
synth_temporal_std = synth_embs.std(axis=2).mean()
print(f'  Real temporal std  : {real_temporal_std:.4f}')
print(f'  Synth temporal std : {synth_temporal_std:.4f}')

mean_cos = np.mean(cos_sims)
if mean_cos > 0.8 and 0.5 < ratio < 2.0:
    print('\n✅ SANITY CHECK PASSED — Safe to run full generation.')
elif mean_cos > 0.6:
    print('\n⚠️  BORDERLINE — Consider reducing ALPHA.')
else:
    print('\n❌ SANITY CHECK FAILED — Reduce ALPHA to 0.3–0.5 and retry.')

---
## CELL 6 — Visualize Single Group: Real vs Synthetic

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle(f'Real vs Synthetic Frame-Level Embeddings — {test_key}',
             fontsize=13, fontweight='bold')

# Row 1: Real sample heatmaps
for i in range(3):
    im = axes[0][i].imshow(real_embs[i, :, :64].T, aspect='auto',
                            cmap='RdBu_r', origin='lower')
    axes[0][i].set_title(f'Real Sample {i+1}')
    axes[0][i].set_xlabel('Time Frame')
    axes[0][i].set_ylabel('HuBERT Dim (first 64)')

# Row 2: Synthetic sample heatmaps
for i in range(3):
    im = axes[1][i].imshow(synth_embs[i, :, :64].T, aspect='auto',
                            cmap='RdBu_r', origin='lower')
    axes[1][i].set_title(f'Synthetic Sample {i+1}')
    axes[1][i].set_xlabel('Time Frame')
    axes[1][i].set_ylabel('HuBERT Dim (first 64)')

plt.tight_layout()
plt.savefig('./outputs/real_vs_synth_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

# Also plot mean trajectory comparison
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 4))
fig2.suptitle('Mean Trajectory: Real vs Synthetic (first 32 dims)', fontsize=12)

real_mean_traj  = real_embs.mean(axis=0)   # (128, 768)
synth_mean_traj = synth_embs.mean(axis=0)  # (128, 768)

axes2[0].plot(real_mean_traj[:, :32], alpha=0.3, color='crimson')
axes2[0].set_title('Real Mean Trajectory (5 samples averaged)')
axes2[0].set_xlabel('Time Frame'); axes2[0].set_ylabel('Value')

axes2[1].plot(synth_mean_traj[:, :32], alpha=0.3, color='cornflowerblue')
axes2[1].set_title('Synthetic Mean Trajectory (35 samples averaged)')
axes2[1].set_xlabel('Time Frame'); axes2[1].set_ylabel('Value')

plt.tight_layout()
plt.savefig('./outputs/mean_trajectory_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plots saved to outputs/')

---
## CELL 7 — Full Generation Loop

**Skips groups that already have synthetic files** to avoid redoing work on restart.

In [ ]:
all_synth_embeddings = []
all_synth_labels     = []
generation_log       = []
skipped              = 0
t_start              = time.time()

print(f'Generating {N_SYNTHETIC_PER_GROUP} synthetic embeddings per group...')
print(f'Total groups: {len(group_embeddings)}')
print(f'Skipping groups that already have synthetic files.\n')

for (actor, sentence, emotion), real_embs in tqdm(group_embeddings.items(), desc='Generating'):
    fname = SYNTH_EMB_DIR / f'synth_actor{actor}_sent{sentence:02d}_{emotion}.npy'

    # Skip if already generated
    if fname.exists():
        synth = np.load(fname)  # (35, 128, 768)
        skipped += 1
    else:
        synth = generate_synthetic_embeddings_framelevel(
            real_embs, N_SYNTHETIC_PER_GROUP, ALPHA, GAUSSIAN_FRACTION, REG_COVAR
        )
        np.save(fname, synth)

    all_synth_embeddings.append(synth)
    for i in range(len(synth)):
        all_synth_labels.append({'actor': actor, 'sentence': sentence,
                                  'emotion': emotion, 'synth_idx': i})

    # Quality log
    real_mean_f  = real_embs.reshape(len(real_embs), -1).mean(axis=0)
    synth_flat   = synth.reshape(len(synth), -1)
    cos_sims     = [1 - cosine(s, real_mean_f) for s in synth_flat]
    generation_log.append({'actor': actor, 'sentence': sentence, 'emotion': emotion,
                             'mean_cosine': np.mean(cos_sims), 'min_cosine': np.min(cos_sims)})

# Stack all: (640*35, 128, 768)
synth_matrix = np.concatenate(all_synth_embeddings, axis=0)
synth_labels = pd.DataFrame(all_synth_labels)
log_df       = pd.DataFrame(generation_log)

np.save(SYNTH_EMB_DIR / 'all_synthetic_matrix.npy', synth_matrix)
synth_labels.to_csv(SYNTH_EMB_DIR / 'all_synthetic_labels.csv', index=False)
log_df.to_csv('./outputs/generation_quality_log.csv', index=False)

t_elapsed = time.time() - t_start
print(f'\n✅ Generation complete!')
print(f'   Groups processed  : {len(group_embeddings)}')
print(f'   Groups skipped    : {skipped} (already existed)')
print(f'   Synth matrix shape: {synth_matrix.shape}  ← (N_total, 128, 768)')
print(f'   Time elapsed      : {t_elapsed:.1f}s')
print(f'   Mean cosine sim   : {log_df["mean_cosine"].mean():.4f}')

---
## CELL 8 — Quality Report: Cosine Similarity by Emotion

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=True)
fig.suptitle('Cosine Similarity (Synthetic vs Real Mean) by Emotion',
             fontsize=14, fontweight='bold')
palette = sns.color_palette('muted', len(EMOTIONS))

for idx, emotion in enumerate(EMOTIONS):
    ax   = axes[idx // 4][idx % 4]
    data = log_df[log_df['emotion'] == emotion]['mean_cosine']
    ax.hist(data, bins=15, color=palette[idx], alpha=0.85, edgecolor='white')
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean={data.mean():.3f}')
    ax.set_title(emotion.capitalize())
    ax.set_xlabel('Mean Cosine Similarity')
    ax.set_ylabel('Groups')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('./outputs/cosine_sim_by_emotion.png', dpi=150, bbox_inches='tight')
plt.show()

summary = log_df.groupby('emotion')['mean_cosine'].agg(['mean','min','max','std'])
summary.columns = ['Mean Cosine', 'Min Cosine', 'Max Cosine', 'Std']
print('\nCosine Similarity Summary:')
print(summary.to_string(float_format='{:.4f}'.format))

low_q = log_df[log_df['mean_cosine'] < 0.7]
if len(low_q) == 0:
    print('\n✅ All groups cosine ≥ 0.7.')
else:
    print(f'\n⚠️  {len(low_q)} groups have cosine < 0.7.')
    print(low_q[['actor','sentence','emotion','mean_cosine']].to_string(index=False))

print('\n✅ Notebook 2 complete. Proceed to Notebook 3.')